In [ ]:
import pandas as pd
import os
import json
import re
import ast
from collections import Counter

def parse_postgres_array(val):
    """Convierte cadenas de array de Postgres '{""A"",""B""}' en listas de Python"""
    if not isinstance(val, str) or not val.startswith('{'):
        return val
    # Limpiar llaves y comillas dobles escapadas
    clean = val.strip('{}').replace('""', '"')
    # Si está vacío
    if not clean: return []
    # Intentar parsear como CSV simple (esto es una simplificación)
    try:
        # Si tiene comillas, es más complejo, pero para estos datos basta con esto:
        items = [i.strip('"') for i in clean.split(",")]
        return items
    except: return val

def parse_json_column(val):
    """Convierte strings JSON en diccionarios"""
    if isinstance(val, str) and (val.startswith('{') and val.endswith('}')) and not val.startswith('{"'): 
        # Si es un array de postgres ya lo manejamos arriba, pero aquí buscamos JSON real
        pass
    if isinstance(val, str):
        try: return json.loads(val)
        except: return val
    return val

### 📊 Carga de Datos Simulados (2,000 perfiles)
Cargamos el archivo CSV y aplicamos limpieza para los formatos de Postgres (arrays) y JSONB.

In [ ]:
csv_path = 'perfiles/perfiles_simulados_2000.csv'
df = pd.read_csv(csv_path)

# Aplicar parsers a las columnas específicas
array_columns = ['interests', 'lifestyle_tags', 'profile_images']
json_columns = ['exclusion_rules', 'importance_weights']

for col in array_columns:
    if col in df.columns: df[col] = df[col].apply(parse_postgres_array)

for col in json_columns:
    if col in df.columns: df[col] = df[col].apply(parse_json_column)

print(f"✅ Cargados {len(df)} registros desde el CSV.")
df.head(3)

### 🏆 Análisis de Categorías Relacionadas con el Algoritmo
Visualizamos el estado de las nuevas variables de presupuesto.

In [ ]:
if 'min_budget' in df.columns:
    print("--- Estadísticas de Presupuesto ---")
    stats = df[['min_budget', 'monthly_budget', 'max_budget']].describe()
    print(stats)

### 🎨 Análisis de Intereses y Lifestyle
Frecuencia de las etiquetas de estilo de vida.

In [ ]:
def analizar_columna_lista(columna):
    todas = []
    for lista in df[columna].dropna():
        if isinstance(lista, list):
            todas.extend(lista)
    
    conteo = pd.Series(todas).value_counts()
    return conteo

if 'lifestyle_tags' in df.columns:
    tags_stats = analizar_columna_lista('lifestyle_tags')
    print(f"Número total de categorías únicas: {len(tags_stats)}")
    print("\n--- Conteo de Etiquetas ---")
    print(tags_stats.head(10))